# 00 — Hay teacher audit and smoke test

This notebook audits the **actual instantiated Hay L5PC teacher** used by `neuron_as_deep_net`. It does not alter morphology, discretization, mechanisms, biophysical parameters, or canonical synapse parameters. It uses NEURON only (not CoreNEURON) and writes a small diagnostic bundle to `artifacts/teacher_audit/`.

Run this notebook in a fresh Linux kernel. It is self-contained: the first operational cell clones the owned `Zagred47/giada` repository when it is not already mounted, then checks out the pinned teacher and prepares the NEURON runtime. Set `HAYFLOW_ELM_REPO` or `HAYFLOW_TEACHER_REPO` only for existing checkouts in non-standard locations. The audit fails explicitly if installation or mechanism compilation does not produce a valid runtime.

## 0 — Clone the owned repository first

This is deliberately the first executable operation. An existing checkout supplied through `HAYFLOW_ELM_REPO` (or the current working tree) is preserved; otherwise the notebook clones `Zagred47/giada` and checks out `HAYFLOW_ELM_REF`, defaulting to `main`.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

ELM_REPOSITORY = "https://github.com/Zagred47/giada.git"
ELM_REF = os.environ.get("HAYFLOW_ELM_REF", "main")
if Path("/kaggle/working").is_dir():
    NOTEBOOK_ROOT = Path("/kaggle/working")
elif Path("/content").is_dir():
    NOTEBOOK_ROOT = Path("/content")
else:
    NOTEBOOK_ROOT = Path.cwd().resolve()
WORKSPACE = NOTEBOOK_ROOT / "hayflow_workspace"
WORKSPACE.mkdir(parents=True, exist_ok=True)

def run(command, cwd=None):
    print("+", " ".join(map(str, command)))
    subprocess.run(list(map(str, command)), cwd=cwd, check=True)

elm_override = os.environ.get("HAYFLOW_ELM_REPO")
mounted_candidates = []
if elm_override:
    mounted_candidates.append(Path(elm_override).expanduser())
mounted_candidates.extend([Path.cwd(), *Path.cwd().parents])
mounted_elm = next((
    path.resolve() for path in mounted_candidates
    if (path / "src" / "hayflow_teacher").is_dir()
), None)

if mounted_elm is not None:
    ELM_REPO = mounted_elm
    print("Using mounted owned repository:", ELM_REPO)
else:
    ELM_REPO = WORKSPACE / "elmneuron"
    if not (ELM_REPO / ".git").is_dir():
        run(["git", "clone", ELM_REPOSITORY, ELM_REPO])
    run(["git", "fetch", "origin"], cwd=ELM_REPO)
    run(["git", "checkout", ELM_REF], cwd=ELM_REPO)

if not (ELM_REPO / "src" / "hayflow_teacher").is_dir():
    raise RuntimeError(
        f"{ELM_REPO} does not contain HayFlow; select the correct ref"
    )
os.environ["HAYFLOW_ELM_REPO"] = str(ELM_REPO)
print("Owned repository ready:", ELM_REPO)

### Pin the upstream teacher and prepare NEURON

The owned repository is already available before this step. Now clone/check out the exact teacher commit, install the pinned NEURON 8.x runtime, and compile the original `.mod` files only when compiled artifacts are absent. No source mechanism is edited.

In [ ]:
import shutil

TEACHER_REPOSITORY = (
    "https://github.com/SelfishGene/neuron_as_deep_net.git"
)
TEACHER_COMMIT = "074c4666300a8ad246601dab179a97a6942f0f29"
teacher_override = os.environ.get("HAYFLOW_TEACHER_REPO")
sibling_teacher = ELM_REPO.parent / "neuron_as_deep_net"
if teacher_override:
    TEACHER_REPO = Path(teacher_override).expanduser().resolve()
elif (sibling_teacher / ".git").is_dir():
    TEACHER_REPO = sibling_teacher.resolve()
else:
    TEACHER_REPO = WORKSPACE / "neuron_as_deep_net"
if not (TEACHER_REPO / ".git").is_dir():
    run(["git", "clone", TEACHER_REPOSITORY, TEACHER_REPO])
run(["git", "checkout", "--detach", TEACHER_COMMIT], cwd=TEACHER_REPO)
teacher_head = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=TEACHER_REPO, text=True
).strip()
assert teacher_head == TEACHER_COMMIT, (teacher_head, TEACHER_COMMIT)
os.environ["HAYFLOW_TEACHER_REPO"] = str(TEACHER_REPO)

NEURON_VERSION = "8.2.7"
run([
    sys.executable, "-m", "pip", "install", "--quiet",
    f"neuron=={NEURON_VERSION}", "numpy", "pandas", "matplotlib",
])
SIMULATION_DIR = TEACHER_REPO / "L5PC_NEURON_simulation"
MOD_DIR = SIMULATION_DIR / "mods"
if not MOD_DIR.is_dir():
    raise FileNotFoundError(f"teacher MOD directory missing: {MOD_DIR}")
compiled_candidates = [
    SIMULATION_DIR / arch / relative
    for arch in ("x86_64", "aarch64")
    for relative in ("special", "libnrnmech.so", ".libs/libnrnmech.so")
]
if not any(path.is_file() for path in compiled_candidates):
    nrnivmodl = shutil.which("nrnivmodl")
    if nrnivmodl is None:
        candidate = Path(sys.executable).parent / "nrnivmodl"
        nrnivmodl = str(candidate) if candidate.is_file() else None
    if nrnivmodl is None or shutil.which("gcc") is None:
        raise RuntimeError(
            "NEURON mechanisms require nrnivmodl and a C compiler"
        )
    run([nrnivmodl, "mods"], cwd=SIMULATION_DIR)
if not any(path.is_file() for path in compiled_candidates):
    raise RuntimeError("nrnivmodl produced no recognized mechanism artifact")
print("Teacher repository ready:", TEACHER_REPO)
print("NEURON mechanisms ready:", next(
    path for path in compiled_candidates if path.is_file()
))

## 1 — Environment and reproducibility

Resolve both repositories, verify the pinned upstream commit, load the compiled mechanisms, seed Python/NumPy/NEURON, activate the generator's canonical CVODE mode, and save `environment.json`.

In [ ]:
import json
import os
import sys
from pathlib import Path
from IPython.display import display

start = Path.cwd().resolve()
elm_candidates = []
if os.environ.get("HAYFLOW_ELM_REPO"):
    elm_candidates.append(Path(os.environ["HAYFLOW_ELM_REPO"]).expanduser())
elm_candidates.extend([start, *start.parents])
elm_candidates.extend([
    start / "elmneuron",
    Path("/kaggle/working/hayflow_workspace/elmneuron"),
    Path("/content/hayflow_workspace/elmneuron"),
])
ELM_REPO = next((
    path.resolve() for path in elm_candidates
    if (path / "src" / "hayflow_teacher").is_dir()
), None)
if ELM_REPO is None:
    raise FileNotFoundError("elmneuron not found; set HAYFLOW_ELM_REPO")
sys.path.insert(0, str(ELM_REPO))

from src.hayflow_teacher import (
    TeacherAuditSession,
    resolve_audit_repositories,
)

ELM_REPO, TEACHER_REPO = resolve_audit_repositories(start)
session = TeacherAuditSession(ELM_REPO, TEACHER_REPO, seed=1729)
environment = session.audit_environment()
display(environment)
print("Artifacts:", session.artifact_dir)

## 2 — Canonical teacher loading

Load the exact morphology, HOC biophysics, and template referenced by the upstream generator. The generator is intentionally not imported: its top level launches the full 128-run dataset job. Instead, its three authoritative synapse factory definitions are extracted from the pinned source AST and executed verbatim. The provenance and this assumption are saved.

In [ ]:
teacher = session.load_canonical_teacher()
display(teacher)
assert teacher["presence"]["soma"]
assert teacher["presence"]["ais"]
assert teacher["presence"]["axon"]
print("Selected variant:", teacher["variant"])
print("Configuration differences:")
display(teacher["variants"])

## 3 — Full morphology audit

Create one row per **instantiated NEURON segment**, including soma and axon. Validate the single-root parent graph and save `segments.csv`. The upstream 639-element dendritic ordering is retained only as provenance for legacy tuft labels; it is not treated as the complete tree.

In [ ]:
segments, morphology = session.audit_morphology()
display(segments.head())
display(segments.tail())
display(morphology)
assert morphology["topology"]["acyclic"]
assert morphology["topology"]["parent_indices_valid"]
assert morphology["topology"]["root_id"] is not None
assert morphology["segment_count"] > morphology["dendritic_segment_count"]

## 4 — Density mechanisms and observable variables

Instantiate the canonical runtime synapses, enumerate density mechanisms and actually test reference accessibility. Inaccessible variables are recorded as `N/R`; they do not abort the audit. Save the per-segment `mechanisms.csv`, aggregate `mechanisms_aggregate.csv`, and the full `teacher_manifest.json`.

In [ ]:
mechanisms, mechanism_summary, synapses = (
    session.audit_mechanisms_and_synapses()
)
display(mechanisms.head())
display(mechanism_summary)
print("Unique mechanisms:", len(mechanism_summary))
print("Manifest variables:", len(session.manifest.variables))

## 5 — Canonical synapse audit

The original generator places one excitatory and one inhibitory point process on every dendritic segment. `ProbAMPANMDA2` is reported as a combined AMPA+NMDA synapse with voltage/magnesium-dependent NMDA, never as NMDA-only. Save `synapses.csv`.

In [ ]:
display(synapses.head())
display(synapses.groupby(["neuron_class", "functional_type"]).size())
assert set(synapses["neuron_class"]) == {
    "ProbAMPANMDA2", "ProbUDFsyn2"
}
combined = synapses[synapses["neuron_class"] == "ProbAMPANMDA2"]
assert combined["components"].str.contains("AMPA").all()
assert combined["components"].str.contains("NMDA").all()

## 6 — No-input smoke test

Run canonical `finitialize(-76)` and 40 ms with no scheduled input. Record soma, AIS, nexus, basal, trunk, and tuft voltages; check finite values, monotonic time, plausible bounds, and unexpected somatic spikes. Save `smoke_test_rest.npz` and PNG.

In [ ]:
rest_report = session.smoke_test_rest(duration_ms=40.0)
display(rest_report)

## 7 — Minimal active smoke test

Use only the upstream-supported `NetCon.event` protocol: one basal combined AMPA+NMDA event and a five-event nexus burst. Record all representative voltages and the exact presynaptic event list. Save `smoke_test_active.npz` and PNG.

In [ ]:
active_report = session.smoke_test_active(duration_ms=50.0)
display(active_report)
print("Basal response ΔV (mV):", active_report["basal_response_delta_mv"])
print("Nexus response ΔV (mV):", active_report["nexus_response_delta_mv"])

## 8 — Internal-state access test

For soma, AIS, basal, trunk, nexus, and tuft, attempt a short recording of all discoverable STATE variables, ionic currents, concentrations, and local excitatory synaptic conductances. Save names, accessibility, and short numeric summaries to `state_variables.json`.

In [ ]:
state_report = session.audit_state_access(duration_ms=5.0)
display({key: value for key, value in state_report.items() if key != "variables"})
state_table = session.pd.DataFrame(state_report["variables"])
display(state_table.groupby(["scope", "kind", "access"]).size())
display(state_table[state_table["access"] == "N/R"].head(20))

## 9 — Snapshot and restore

Checkpoint with `SaveState`, deliver the same canonical nexus event in two continuations, and compare interpolated voltage traces and somatic spikes. The upstream point processes leave their RNG pointer unbound and use a global legacy `exprand`; the audit therefore resets `h.set_seed` before each branch and records that production limitation explicitly.

In [ ]:
snapshot_report = session.snapshot_restore()
display(snapshot_report)

## 10 — Artifact contract

Verify the required machine-readable outputs. The final cell also writes `final_report.json` and `artifact_index.json` with relative names, hashes, and sizes.

In [ ]:
required_outputs = [
    "environment.json",
    "teacher_manifest.json",
    "segments.csv",
    "mechanisms.csv",
    "synapses.csv",
    "state_variables.json",
    "smoke_test_rest.npz",
    "smoke_test_active.npz",
    "snapshot_restore_report.json",
    "smoke_test_rest.png",
    "smoke_test_active.png",
]
artifact_rows = []
for name in required_outputs:
    path = session.artifact_dir / name
    artifact_rows.append({
        "path": name,
        "exists": path.is_file(),
        "size_bytes": path.stat().st_size if path.is_file() else None,
    })
artifact_table = session.pd.DataFrame(artifact_rows)
display(artifact_table)
assert artifact_table["exists"].all()

## 11 — Final report

Print and save the requested decision-oriented summary: loaded teacher, complete morphology, synapses, observable states/calcium, snapshot determinism, blockers, and recommended next action.

In [ ]:
final_report = session.finalize()
print(json.dumps(final_report, indent=2, sort_keys=True))
print("\nAudit complete:", session.artifact_dir)